In [ ]:
import pandas as pd
from pathlib import Path


# ============================================================
# STEP 10 - BUILD COMPLETE ATTENDANCE REQUIRED MASTER
# ============================================================
#
# OUTPUT GRAIN:
#   dim_employee       : 1 row per employee_id
#   fact_attendance_daily : 1 row per employee_id + attendance_date
#   attendance_required: 1 employee x 1 calendar date
#   fact_employee_week : 1 employee_id x 1 week_start
#
# MAIN LOGIC:
#   1. Load attendance and calendar.
#   2. Validate and standardize source data.
#   3. Build employee master using ALL available employee attributes.
#   4. Aggregate attendance transactions to employee-date level.
#   5. Create employee x calendar grid.
#   6. LEFT JOIN daily attendance to the grid.
#   7. Calculate daily and weekly attendance requirements.
#   8. Validate and export.
#
# IMPORTANT:
#   has_clock_in            = from Step 8
#   has_clock_out           = from Step 8
#   has_attendance_activity = from Step 8
#
# has_attendance_activity is NOT defined by clock flags.
# It means that an attendance source record exists.
#
# ============================================================


# ============================================================
# 1. CONFIGURATION
# ============================================================

ATTENDANCE_FILE = Path(
    "output/second_reports/attendance_feature_engineered.xlsx"
)

CALENDAR_FILE = Path(
    "output/dim_calendar.parquet"
)

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Employee master
EMPLOYEE_PARQUET = OUTPUT_DIR / "dim_employee.parquet"
EMPLOYEE_CSV = OUTPUT_DIR / "dim_employee.csv"

# Main employee-date table
REQUIRED_PARQUET = OUTPUT_DIR / "attendance_required.parquet"
REQUIRED_CSV = OUTPUT_DIR / "attendance_required.csv"

# Daily fact
DAILY_PARQUET = OUTPUT_DIR / "fact_attendance_daily.parquet"
DAILY_CSV = OUTPUT_DIR / "fact_attendance_daily.csv"

# Weekly fact
WEEKLY_PARQUET = OUTPUT_DIR / "fact_employee_week.parquet"
WEEKLY_CSV = OUTPUT_DIR / "fact_employee_week.csv"

# Employee attribute conflict report
CONFLICT_CSV = OUTPUT_DIR / "employee_attribute_conflicts.csv"


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def to_boolean(series):
    """
    Convert common Excel/Pandas boolean representations.

    Missing values are treated as False.
    Unexpected values raise an error.
    """

    true_values = {
        "TRUE", "1", "YES", "Y", "T"
    }

    false_values = {
        "FALSE", "0", "NO", "N", "F",
        "", "NONE", "NAN", "<NA>"
    }

    def convert(value):

        if pd.isna(value):
            return False

        if isinstance(value, bool):
            return value

        value_text = str(value).strip().upper()

        if value_text in true_values:
            return True

        if value_text in false_values:
            return False

        raise ValueError(
            f"Unexpected boolean value: {value!r}"
        )

    return series.map(convert).astype(bool)


def first_non_null(series):
    """
    Return the first non-null value.
    Used when aggregating daily attendance details.
    """

    values = series.dropna()

    if values.empty:
        return pd.NA

    return values.iloc[0]


def most_frequent_non_null(series):
    """
    Select the most frequent non-null employee attribute.

    If multiple values have the same frequency,
    select the last observed value among those candidates.
    """

    values = series.dropna()

    if values.empty:
        return pd.NA

    counts = values.value_counts(dropna=True)

    highest_count = counts.max()

    candidates = counts[
        counts == highest_count
    ].index.tolist()

    for value in reversed(values.tolist()):

        if value in candidates:
            return value

    return values.iloc[-1]


def validate_required_columns(df, required_columns, source_name):

    missing_columns = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{source_name} is missing required columns: "
            f"{missing_columns}"
        )


# ============================================================
# 3. LOAD INPUT DATA
# ============================================================

print("=" * 60)
print("STEP 10 - BUILD COMPLETE ATTENDANCE REQUIRED MASTER")
print("=" * 60)

print("\nLoading source data...")

if not ATTENDANCE_FILE.exists():
    raise FileNotFoundError(
        f"Attendance file not found: {ATTENDANCE_FILE}"
    )

if not CALENDAR_FILE.exists():
    raise FileNotFoundError(
        f"Calendar file not found: {CALENDAR_FILE}"
    )

attendance = pd.read_excel(
    ATTENDANCE_FILE
)

calendar = pd.read_parquet(
    CALENDAR_FILE
)

print("Attendance source rows:", len(attendance))
print("Calendar rows:", len(calendar))
print("Attendance source columns:", len(attendance.columns))


# ============================================================
# 4. VALIDATE REQUIRED INPUT COLUMNS
# ============================================================

attendance_required_columns = [
    "employee_id",
    "attendance_date",
    "cost_center_type",
    "weekly_work_quota",
    "has_clock_in",
    "has_clock_out",
    "has_attendance_activity"
]

calendar_required_columns = [
    "attendance_date",
    "week_start",
    "week_end",
    "is_weekend",
    "is_national_holiday",
    "is_joint_leave",
    "weekday_national_holidays",
    "weekday_joint_leaves"
]

validate_required_columns(
    attendance,
    attendance_required_columns,
    "Attendance source"
)

validate_required_columns(
    calendar,
    calendar_required_columns,
    "Calendar source"
)

print("\nRequired input columns: VALID")


# ============================================================
# 5. STANDARDIZE EMPLOYEE ID AND DATE COLUMNS
# ============================================================

print("\nStandardizing keys and dates...")

# Employee ID
attendance["employee_id"] = (
    attendance["employee_id"]
    .astype("string")
    .str.strip()
)

# Attendance date
attendance["attendance_date"] = pd.to_datetime(
    attendance["attendance_date"],
    errors="coerce"
).dt.normalize()

# Calendar dates
for col in [
    "attendance_date",
    "week_start",
    "week_end"
]:

    calendar[col] = pd.to_datetime(
        calendar[col],
        errors="coerce"
    ).dt.normalize()


# Employment dates are optional.
# They are used to restrict employee-calendar combinations.
for col in [
    "employee_start_date",
    "employee_end_date"
]:

    if col in attendance.columns:

        attendance[col] = pd.to_datetime(
            attendance[col],
            errors="coerce"
        ).dt.normalize()


# Validate employee ID
if attendance["employee_id"].isna().any():

    raise ValueError(
        "Missing employee_id found in attendance source."
    )

if attendance["employee_id"].eq("").any():

    raise ValueError(
        "Empty employee_id found in attendance source."
    )


# Validate attendance date
if attendance["attendance_date"].isna().any():

    invalid_rows = attendance[
        attendance["attendance_date"].isna()
    ]

    print(invalid_rows.head(10))

    raise ValueError(
        "Invalid attendance_date found in attendance source."
    )


# Validate calendar dates
if calendar["attendance_date"].isna().any():

    raise ValueError(
        "Invalid attendance_date found in calendar."
    )

if calendar["week_start"].isna().any():

    raise ValueError(
        "Missing or invalid week_start in calendar."
    )

if calendar["week_end"].isna().any():

    raise ValueError(
        "Missing or invalid week_end in calendar."
    )

if calendar["attendance_date"].duplicated().any():

    raise ValueError(
        "Calendar must contain exactly one row per attendance_date."
    )

print("Key and date validation: VALID")


# ============================================================
# 6. STANDARDIZE BOOLEAN FLAGS
# ============================================================

# These flags were already created in Step 8.
# Step 10 must use the existing definitions.

attendance_flag_columns = [
    "has_clock_in",
    "has_clock_out",
    "has_attendance_activity"
]

for col in attendance_flag_columns:

    attendance[col] = to_boolean(
        attendance[col]
    )


calendar_flag_columns = [
    "is_weekend",
    "is_national_holiday",
    "is_joint_leave"
]

for col in calendar_flag_columns:

    calendar[col] = to_boolean(
        calendar[col]
    )


# ============================================================
# 7. STANDARDIZE COST CENTER AND WEEKLY QUOTA
# ============================================================

attendance["cost_center_type"] = (
    attendance["cost_center_type"]
    .astype("string")
    .str.strip()
    .str.upper()
    .fillna("UNKNOWN")
)

attendance["weekly_work_quota"] = pd.to_numeric(
    attendance["weekly_work_quota"],
    errors="coerce"
).astype("Int64")


if "schedule_type" in attendance.columns:

    attendance["schedule_type"] = (
        attendance["schedule_type"]
        .astype("string")
        .str.strip()
        .str.upper()
        .fillna("UNKNOWN")
    )


valid_cost_center_types = {
    "FIXED_5",
    "NON_FIXED_6",
    "UNKNOWN"
}

unexpected_types = (
    set(attendance["cost_center_type"].dropna().unique())
    - valid_cost_center_types
)

if unexpected_types:

    raise ValueError(
        "Unexpected cost_center_type values: "
        f"{sorted(unexpected_types)}"
    )


# Validate quota assignment
invalid_fixed_quota = (
    attendance["cost_center_type"].eq("FIXED_5")
    & attendance["weekly_work_quota"].ne(5)
)

invalid_non_fixed_quota = (
    attendance["cost_center_type"].eq("NON_FIXED_6")
    & attendance["weekly_work_quota"].ne(6)
)

if invalid_fixed_quota.any():

    raise ValueError(
        "FIXED_5 employees must have weekly_work_quota = 5."
    )

if invalid_non_fixed_quota.any():

    raise ValueError(
        "NON_FIXED_6 employees must have weekly_work_quota = 6."
    )


print("Cost center classification: VALID")
print("Weekly quota assignment: VALID")


# ============================================================
# 8. BUILD EMPLOYEE MASTER
# ============================================================

print("\nBuilding employee master...")


# These columns describe individual attendance events
# or calendar-level information.
#
# They should NOT be aggregated as employee attributes.

attendance_event_columns = {

    # Attendance date and time
    "attendance_date",
    "clock_in_time",
    "clock_out_time",
    "clock_in_time_be",
    "clock_out_time_be",

    # Attendance information
    "attendance_in",
    "attendance_out",
    "attendance_status",

    # Shift information
    "shift_in",
    "shift_out",
    "shift_code",

    # Attendance flags
    "has_clock_in",
    "has_clock_out",
    "has_attendance_activity",

    # Calendar information
    "year_week",
    "week_start",
    "week_end",
    "is_weekend",
    "is_national_holiday",
    "is_joint_leave",
    "weekday_national_holidays",
    "weekday_joint_leaves",

    # Daily attendance features
    "attendance_record_count",
    "attendance_record_status",
    "is_expected_workday",
    "daily_work_requirement",
    "expected_workday_attended",
    "attendance_outside_expected_day"
}


# All other columns are treated as employee attributes.
#
# This keeps available employee information dynamically,
# including department, division, organization, location,
# cost center, store, position, employee dates, etc.
#
# No fixed short list of employee attributes is used.

employee_attribute_columns = [

    col
    for col in attendance.columns

    if col not in attendance_event_columns
    and col != "employee_id"

]


print(
    "Employee attributes detected:",
    len(employee_attribute_columns)
)

print(
    "Employee attribute columns:",
    employee_attribute_columns
)


# Aggregate employee attributes.
# Select the most frequent non-null value per employee.

employee_aggregation = {

    col: most_frequent_non_null

    for col in employee_attribute_columns

}


employee_master = (

    attendance[
        ["employee_id"] + employee_attribute_columns
    ]

    .groupby(
        "employee_id",
        as_index=False,
        dropna=False
    )

    .agg(employee_aggregation)

)


# Ensure one row per employee.
if employee_master["employee_id"].duplicated().any():

    raise ValueError(
        "Duplicate employee_id found in employee master."
    )


# Normalize employee dates after aggregation.
for col in [
    "employee_start_date",
    "employee_end_date"
]:

    if col in employee_master.columns:

        employee_master[col] = pd.to_datetime(
            employee_master[col],
            errors="coerce"
        ).dt.normalize()


print("Employee master created.")
print("Unique employees:", len(employee_master))
print("Employee attributes retained:", len(employee_attribute_columns))


# ============================================================
# 9. REPORT EMPLOYEE ATTRIBUTE CONFLICTS
# ============================================================

print("\nChecking employee attribute consistency...")

conflict_rows = []

for col in employee_attribute_columns:

    counts = (

        attendance.groupby("employee_id")[col]

        .nunique(dropna=True)

    )

    conflicting_ids = counts[
        counts > 1
    ]

    if not conflicting_ids.empty:

        conflict_rows.append({

            "column": col,

            "employees_with_multiple_values": int(
                len(conflicting_ids)
            )

        })


employee_conflicts = pd.DataFrame(
    conflict_rows,
    columns=[
        "column",
        "employees_with_multiple_values"
    ]
)


if not employee_conflicts.empty:

    print("\nWARNING: Employee attribute conflicts found.")

    print(
        employee_conflicts.to_string(index=False)
    )

    employee_conflicts.to_csv(
        CONFLICT_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "\nThe employee master uses the most frequent "
        "non-null value for each employee."
    )

    print(
        "Review department, division, organization, "
        "location, and cost center conflicts."
    )

else:

    print("Employee attribute consistency: VALID")


# ============================================================
# 10. BUILD DAILY ATTENDANCE FACT
# ============================================================

print("\nBuilding daily attendance fact...")


# These are event-level columns that should remain available
# in the daily attendance output.

daily_detail_candidates = [

    "clock_in_time",
    "clock_out_time",
    "clock_in_time_be",
    "clock_out_time_be",

    "attendance_in",
    "attendance_out",

    "shift_in",
    "shift_out",
    "shift_code",

    "attendance_status"

]


daily_detail_columns = [

    col

    for col in daily_detail_candidates

    if col in attendance.columns

]


# Aggregate all attendance transactions into one employee-date row.

daily_aggregation = {

    "attendance_record_count": (
        "attendance_date",
        "size"
    ),

    "has_clock_in": (
        "has_clock_in",
        "max"
    ),

    "has_clock_out": (
        "has_clock_out",
        "max"
    ),

    "has_attendance_activity": (
        "has_attendance_activity",
        "max"
    )

}


# Retain available attendance detail fields.
for col in daily_detail_columns:

    daily_aggregation[col] = (
        col,
        first_non_null
    )


daily_attendance = (

    attendance.groupby(

        [
            "employee_id",
            "attendance_date"
        ],

        as_index=False,
        dropna=False

    )

    .agg(**daily_aggregation)

)


# Validate daily grain.
if daily_attendance.duplicated(
    ["employee_id", "attendance_date"]
).any():

    raise ValueError(
        "Daily attendance contains duplicate employee-date keys."
    )


print("Daily attendance rows:", len(daily_attendance))


# ============================================================
# 11. PREPARE CALENDAR REFERENCE
# ============================================================

print("\nPreparing calendar reference...")


calendar_columns = [

    "attendance_date",
    "week_start",
    "week_end",

    "is_weekend",
    "is_national_holiday",
    "is_joint_leave",

    "weekday_national_holidays",
    "weekday_joint_leaves"

]


calendar_reference = calendar[
    calendar_columns
].copy()


# Convert holiday count columns to numeric.
for col in [
    "weekday_national_holidays",
    "weekday_joint_leaves"
]:

    calendar_reference[col] = pd.to_numeric(
        calendar_reference[col],
        errors="coerce"
    ).fillna(0).astype("int64")


# Check attendance dates against calendar.
calendar_dates = set(
    calendar_reference["attendance_date"]
)

attendance_dates = set(
    daily_attendance["attendance_date"]
)

unmatched_dates = (
    attendance_dates - calendar_dates
)

if unmatched_dates:

    raise ValueError(
        "Attendance dates not found in calendar. "
        f"Examples: {sorted(unmatched_dates)[:10]}"
    )


print("Calendar reference: VALID")


# ============================================================
# 12. CREATE EMPLOYEE x CALENDAR GRID
# ============================================================

print("\nCreating employee x calendar grid...")


# IMPORTANT:
#
# Start from employee_master, not attendance transactions.
#
# This ensures every employee identified in the source
# appears across the calendar dates within their employment
# period, including dates with no attendance transactions.

attendance_required = employee_master.merge(
    calendar_reference,
    how="cross"
)


# Apply employee start date.
if "employee_start_date" in attendance_required.columns:

    attendance_required = attendance_required.loc[

        attendance_required["employee_start_date"].isna()

        |

        (
            attendance_required["attendance_date"]
            >= attendance_required["employee_start_date"]
        )

    ].copy()


# Apply employee end date.
if "employee_end_date" in attendance_required.columns:

    attendance_required = attendance_required.loc[

        attendance_required["employee_end_date"].isna()

        |

        (
            attendance_required["attendance_date"]
            <= attendance_required["employee_end_date"]
        )

    ].copy()


print(
    "Employee-calendar combinations:",
    len(attendance_required)
)


# ============================================================
# 13. LEFT JOIN DAILY ATTENDANCE
# ============================================================

print("\nJoining daily attendance to employee-calendar grid...")


attendance_required = attendance_required.merge(

    daily_attendance,

    on=[
        "employee_id",
        "attendance_date"
    ],

    how="left",

    validate="one_to_one"

)


# Dates without attendance transactions:
# attendance_record_count = 0
# attendance flags = False

attendance_required["attendance_record_count"] = (

    attendance_required["attendance_record_count"]

    .fillna(0)

    .astype("int64")

)


for col in attendance_flag_columns:

    attendance_required[col] = (

        attendance_required[col]

        .fillna(False)

        .astype(bool)

    )


# Explicit record status.
attendance_required["attendance_record_status"] = (

    attendance_required["attendance_record_count"]

    .gt(0)

    .map({

        True: "HAS_ATTENDANCE_RECORD",

        False: "NO_ATTENDANCE_RECORD"

    })

)


print("Employee-date rows:", len(attendance_required))

print(
    "Rows with attendance:",
    int(
        attendance_required[
            "attendance_record_count"
        ].gt(0).sum()
    )
)

print(
    "Rows without attendance:",
    int(
        attendance_required[
            "attendance_record_count"
        ].eq(0).sum()
    )
)


# ============================================================
# 14. VALIDATE ATTENDANCE FLAGS
# ============================================================

print("\nValidating attendance flags...")


# Correct definition:
#
# has_attendance_activity = TRUE
# when an attendance source record exists.
#
# It is NOT required that clock-in or clock-out exists.
#
# A record can exist even when both clock values are empty.

expected_activity = (

    attendance_required["attendance_record_count"] > 0

)


invalid_activity = (

    attendance_required["has_attendance_activity"]

    != expected_activity

)


if invalid_activity.any():

    print(

        attendance_required.loc[

            invalid_activity,

            [

                "employee_id",
                "attendance_date",
                "attendance_record_count",
                "has_clock_in",
                "has_clock_out",
                "has_attendance_activity"

            ]

        ].head(20)

    )

    raise ValueError(

        "has_attendance_activity is inconsistent "
        "with attendance_record_count."

    )


# Clock flags cannot be True when no attendance record exists.
# But attendance activity may exist while both clock flags are False.

invalid_clock_flags = (

    (

        attendance_required["has_clock_in"]

        |

        attendance_required["has_clock_out"]

    )

    &

    ~attendance_required["has_attendance_activity"]

)


if invalid_clock_flags.any():

    raise ValueError(

        "Clock flags are True even though "
        "has_attendance_activity is False."

    )


print("Attendance activity definition: VALID")
print("Clock flag consistency: VALID")


# ============================================================
# 15. CALCULATE DAILY WORK REQUIREMENT
# ============================================================

print("\nCalculating daily work requirements...")


attendance_required["cost_center_type"] = (

    attendance_required["cost_center_type"]

    .astype("string")

    .str.strip()

    .str.upper()

    .fillna("UNKNOWN")

)


attendance_required["is_expected_workday"] = pd.Series(

    pd.NA,

    index=attendance_required.index,

    dtype="boolean"

)


attendance_required["daily_work_requirement"] = pd.Series(

    pd.NA,

    index=attendance_required.index,

    dtype="Int64"

)


is_fixed = (

    attendance_required["cost_center_type"]

    .eq("FIXED_5")

)


is_non_fixed = (

    attendance_required["cost_center_type"]

    .eq("NON_FIXED_6")

)


is_weekday = (

    ~attendance_required["is_weekend"]

)


# FIXED_5:
# Monday-Friday, excluding national holidays.

is_fixed_workday = (

    is_fixed

    & is_weekday

    & ~attendance_required["is_national_holiday"]

)


attendance_required.loc[

    is_fixed,

    "is_expected_workday"

] = is_fixed_workday.loc[is_fixed]


attendance_required.loc[

    is_fixed_workday,

    "daily_work_requirement"

] = 1


attendance_required.loc[

    is_fixed & ~is_fixed_workday,

    "daily_work_requirement"

] = 0


# NON_FIXED_6:
# Individual daily schedule is unknown.
# Weekly quota remains six days.
#
# Therefore is_expected_workday and daily_work_requirement
# remain unknown for NON_FIXED_6 employees.


# UNKNOWN:
# Daily requirement also remains unknown.


# ============================================================
# 16. BUILD WEEKLY CALENDAR REFERENCE
# ============================================================

print("\nBuilding weekly calendar reference...")


weekly_calendar = (

    calendar_reference.groupby(

        "week_start",

        as_index=False

    )

    .agg(

        week_end=(
            "week_end",
            "max"
        ),

        calendar_days=(
            "attendance_date",
            "nunique"
        ),

        weekday_national_holidays=(
            "weekday_national_holidays",
            "max"
        ),

        weekday_joint_leaves=(
            "weekday_joint_leaves",
            "max"
        )

    )

)


weekly_calendar["is_full_calendar_week"] = (

    weekly_calendar["calendar_days"].eq(7)

)


# ============================================================
# 17. AGGREGATE EMPLOYEE-WEEK DATA
# ============================================================

print("\nAggregating employee-week data...")


# Attendance on an expected workday.
attendance_required["expected_workday_attended"] = (

    attendance_required["is_expected_workday"].fillna(False)

    &

    attendance_required["has_attendance_activity"]

)


# Attendance outside expected working days.
attendance_required["attendance_outside_expected_day"] = (

    is_fixed

    &

    ~attendance_required["is_expected_workday"].fillna(False)

    &

    attendance_required["has_attendance_activity"]

)


weekly = (

    attendance_required.groupby(

        [
            "employee_id",
            "week_start"
        ],

        as_index=False,

        dropna=False

    )

    .agg(

        attendance_record_count=(
            "attendance_record_count",
            "sum"
        ),

        employee_week_calendar_days=(
            "attendance_date",
            "nunique"
        ),

        actual_attendance_days=(
            "has_attendance_activity",
            "sum"
        ),

        actual_clock_in_days=(
            "has_clock_in",
            "sum"
        ),

        actual_clock_out_days=(
            "has_clock_out",
            "sum"
        ),

        actual_expected_workdays_attended=(
            "expected_workday_attended",
            "sum"
        ),

        attendance_outside_expected_days=(
            "attendance_outside_expected_day",
            "sum"
        )

    )

)


# Add calendar information.
weekly = weekly.merge(

    weekly_calendar,

    on="week_start",

    how="left",

    validate="many_to_one"

)


# Add ALL employee master attributes.
weekly = weekly.merge(

    employee_master,

    on="employee_id",

    how="left",

    validate="many_to_one"

)


if weekly["calendar_days"].isna().any():

    raise ValueError(
        "Some employee weeks have no calendar reference."
    )


# ============================================================
# 18. IDENTIFY EVALUABLE EMPLOYEE WEEKS
# ============================================================

# A calendar week is only evaluated when:
#
# 1. The calendar contains all seven days.
# 2. The employee has all seven days within their employment period.
#
# This prevents a newly hired or resigned employee from being
# evaluated against a full weekly quota for a partial employment week.

weekly["is_evaluable_week"] = (

    weekly["is_full_calendar_week"]

    &

    weekly["employee_week_calendar_days"].eq(7)

)


full_week = weekly["is_evaluable_week"]


weekly["cost_center_type"] = (

    weekly["cost_center_type"]

    .astype("string")

    .str.strip()

    .str.upper()

    .fillna("UNKNOWN")

)


fixed_mask = (

    weekly["cost_center_type"]

    .eq("FIXED_5")

)


non_fixed_mask = (

    weekly["cost_center_type"]

    .eq("NON_FIXED_6")

)


unknown_mask = (

    weekly["cost_center_type"]

    .eq("UNKNOWN")

)


# ============================================================
# 19. CALCULATE WEEKLY REQUIRED DAYS
# ============================================================

print("\nCalculating weekly required days...")


weekly["weekly_required_days"] = pd.Series(

    pd.NA,

    index=weekly.index,

    dtype="Int64"

)


# FIXED_5:
# Five working days minus weekday national holidays.

fixed_required = (

    5 - weekly["weekday_national_holidays"]

)


if (fixed_required < 0).any():

    raise ValueError(
        "Weekday national holidays exceed the five-day quota."
    )


weekly.loc[

    full_week & fixed_mask,

    "weekly_required_days"

] = fixed_required.loc[

    full_week & fixed_mask

].astype("int64")


# NON_FIXED_6:
# Weekly quota remains six days, including holiday weeks.

weekly.loc[

    full_week & non_fixed_mask,

    "weekly_required_days"

] = 6


# UNKNOWN:
# Weekly required days remain null.


# ============================================================
# 20. CALCULATE WEEKLY ATTENDANCE GAP
# ============================================================

weekly["attendance_gap"] = pd.Series(

    pd.NA,

    index=weekly.index,

    dtype="Int64"

)


# FIXED_5:
# Compare expected working days attended against required days.

fixed_gap = (

    weekly["weekly_required_days"]

    - weekly["actual_expected_workdays_attended"]

)


weekly.loc[

    full_week & fixed_mask,

    "attendance_gap"

] = fixed_gap.loc[

    full_week & fixed_mask

]


# NON_FIXED_6:
# Compare actual attendance days against six-day quota.

non_fixed_gap = (

    weekly["weekly_required_days"]

    - weekly["actual_attendance_days"]

)


weekly.loc[

    full_week & non_fixed_mask,

    "attendance_gap"

] = non_fixed_gap.loc[

    full_week & non_fixed_mask

]


# ============================================================
# 21. CREATE WEEKLY ATTENDANCE STATUS
# ============================================================

weekly["weekly_attendance_status"] = "UNKNOWN"


# Partial calendar or employment week.
weekly.loc[

    ~full_week,

    "weekly_attendance_status"

] = "PARTIAL_WEEK_NOT_EVALUATED"


# Unknown cost center.
weekly.loc[

    full_week & unknown_mask,

    "weekly_attendance_status"

] = "UNKNOWN_COST_CENTER"


# FIXED_5 status.
weekly.loc[

    full_week

    & fixed_mask

    & (weekly["attendance_gap"] <= 0),

    "weekly_attendance_status"

] = "QUOTA_MET_OR_EXCEEDED"


weekly.loc[

    full_week

    & fixed_mask

    & (weekly["attendance_gap"] > 0),

    "weekly_attendance_status"

] = "BELOW_REQUIRED_DAYS"


# NON_FIXED_6 status.
weekly.loc[

    full_week

    & non_fixed_mask

    & (weekly["attendance_gap"] <= 0),

    "weekly_attendance_status"

] = "AT_OR_ABOVE_WEEKLY_QUOTA"


weekly.loc[

    full_week

    & non_fixed_mask

    & (weekly["attendance_gap"] > 0),

    "weekly_attendance_status"

] = "BELOW_WEEKLY_QUOTA_REVIEW"


# ============================================================
# 22. CREATE WEEKLY INDICATORS
# ============================================================

weekly["is_quota_gap"] = (

    weekly["attendance_gap"] > 0

).astype("boolean")


weekly["is_quota_met"] = (

    weekly["attendance_gap"] <= 0

).astype("boolean")


not_evaluated = (

    ~full_week

    |

    unknown_mask

)


weekly.loc[

    not_evaluated,

    [
        "is_quota_gap",
        "is_quota_met"
    ]

] = pd.NA


# ============================================================
# 23. DATA VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("STEP 10 DATA VALIDATION")
print("=" * 60)


# ------------------------------------------------------------
# 23A. Employee master grain
# ------------------------------------------------------------

if employee_master["employee_id"].duplicated().any():

    raise ValueError(
        "Duplicate employee_id in dim_employee."
    )


# ------------------------------------------------------------
# 23B. Employee-date grain
# ------------------------------------------------------------

if attendance_required.duplicated(

    [
        "employee_id",
        "attendance_date"
    ]

).any():

    raise ValueError(
        "Duplicate employee-date rows in attendance_required."
    )


# ------------------------------------------------------------
# 23C. Employee-week grain
# ------------------------------------------------------------

if weekly.duplicated(

    [
        "employee_id",
        "week_start"
    ]

).any():

    raise ValueError(
        "Duplicate employee-week rows in weekly fact."
    )


# ------------------------------------------------------------
# 23D. Employee coverage
# ------------------------------------------------------------

employee_ids_master = set(

    employee_master["employee_id"]

)

employee_ids_required = set(

    attendance_required["employee_id"]

)


missing_employees = (

    employee_ids_master - employee_ids_required

)


if missing_employees:

    raise ValueError(

        "Some employees are missing from attendance_required: "

        f"{list(missing_employees)[:10]}"

    )


# ------------------------------------------------------------
# 23E. Attendance record count
# ------------------------------------------------------------

if attendance_required[
    "attendance_record_count"
].lt(0).any():

    raise ValueError(
        "Negative attendance_record_count found."
    )


# ------------------------------------------------------------
# 23F. Attendance activity consistency
# ------------------------------------------------------------

expected_activity = (

    attendance_required["attendance_record_count"] > 0

)


invalid_activity = (

    attendance_required["has_attendance_activity"]

    != expected_activity

)


if invalid_activity.any():

    raise ValueError(

        "has_attendance_activity is inconsistent "
        "with attendance_record_count."

    )


# ------------------------------------------------------------
# 23G. Clock flag consistency
# ------------------------------------------------------------

invalid_clock_flags = (

    (

        attendance_required["has_clock_in"]

        |

        attendance_required["has_clock_out"]

    )

    &

    ~attendance_required["has_attendance_activity"]

)


if invalid_clock_flags.any():

    raise ValueError(
        "Clock flags are True without attendance activity."
    )


# ------------------------------------------------------------
# 23H. Weekly quota validation
# ------------------------------------------------------------

fixed_quota_values = weekly.loc[

    weekly["cost_center_type"].eq("FIXED_5"),

    "weekly_required_days"

].dropna()


if (fixed_quota_values > 5).any():

    raise ValueError(
        "FIXED_5 weekly quota exceeds five days."
    )


non_fixed_quota_values = weekly.loc[

    weekly["cost_center_type"].eq("NON_FIXED_6"),

    "weekly_required_days"

].dropna()


if (non_fixed_quota_values != 6).any():

    raise ValueError(
        "NON_FIXED_6 weekly quota must equal six days."
    )


# ------------------------------------------------------------
# 23I. Validate employee attributes in weekly output
# ------------------------------------------------------------

missing_employee_attributes = [

    col

    for col in employee_attribute_columns

    if col not in weekly.columns

]


if missing_employee_attributes:

    raise ValueError(

        "Employee attributes missing from weekly output: "

        f"{missing_employee_attributes}"

    )


print("Employee master grain: VALID")
print("Attendance required grain: VALID")
print("Weekly grain: VALID")
print("Employee coverage: VALID")
print("Attendance activity flags: VALID")
print("Clock flag consistency: VALID")
print("Weekly quota rules: VALID")
print("Employee attributes in weekly fact: VALID")


# ============================================================
# 24. EXPORT EMPLOYEE MASTER
# ============================================================

print("\nExporting employee master...")


employee_master.to_parquet(

    EMPLOYEE_PARQUET,

    index=False

)


employee_master.to_csv(

    EMPLOYEE_CSV,

    index=False,

    encoding="utf-8-sig"

)


# ============================================================
# 25. EXPORT ATTENDANCE REQUIRED
# ============================================================

print("Exporting attendance required...")


attendance_required.to_parquet(

    REQUIRED_PARQUET,

    index=False

)


attendance_required.to_csv(

    REQUIRED_CSV,

    index=False,

    encoding="utf-8-sig"

)


# ============================================================
# 26. EXPORT DAILY ATTENDANCE FACT
# ============================================================

print("Exporting daily attendance fact...")


# This output is the same complete employee-date dataset.
# It retains employee attributes, calendar fields,
# attendance detail, and daily work requirement fields.

attendance_required.to_parquet(

    DAILY_PARQUET,

    index=False

)


attendance_required.to_csv(

    DAILY_CSV,

    index=False,

    encoding="utf-8-sig"

)


# ============================================================
# 27. EXPORT WEEKLY FACT
# ============================================================

print("Exporting weekly fact...")


weekly.to_parquet(

    WEEKLY_PARQUET,

    index=False

)


weekly.to_csv(

    WEEKLY_CSV,

    index=False,

    encoding="utf-8-sig"

)


# ============================================================
# 28. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("STEP 10 COMPLETED")
print("=" * 60)


print("\nEMPLOYEE MASTER")

print(
    "Unique employees:",
    len(employee_master)
)

print(
    "Employee attributes retained:",
    len(employee_attribute_columns)
)


print("\nATTENDANCE REQUIRED")

print(
    "Total employee-date rows:",
    len(attendance_required)
)

print(
    "Rows with attendance transaction:",
    int(
        attendance_required[
            "attendance_record_count"
        ].gt(0).sum()
    )
)

print(
    "Rows without attendance transaction:",
    int(
        attendance_required[
            "attendance_record_count"
        ].eq(0).sum()
    )
)


print("\nWEEKLY FACT")

print(
    "Employee-week rows:",
    len(weekly)
)

print(
    "Evaluable employee weeks:",
    int(weekly["is_evaluable_week"].sum())
)

print(
    "Non-evaluable employee weeks:",
    int((~weekly["is_evaluable_week"]).sum())
)


print("\nCOST CENTER DISTRIBUTION")

print(

    weekly["cost_center_type"]

    .value_counts(dropna=False)

)


print("\nWEEKLY REQUIRED DAYS DISTRIBUTION")

print(

    weekly["weekly_required_days"]

    .value_counts(dropna=False)

    .sort_index()

)


print("\nWEEKLY ATTENDANCE STATUS")

print(

    weekly["weekly_attendance_status"]

    .value_counts(dropna=False)

)


print("\nOUTPUT FILES")

print("Employee master:", EMPLOYEE_PARQUET)

print("Attendance required:", REQUIRED_PARQUET)

print("Daily fact:", DAILY_PARQUET)

print("Weekly fact:", WEEKLY_PARQUET)

print("Employee conflict report:", CONFLICT_CSV)


print("\n" + "=" * 60)
print("PROCESS 10 SUCCESSFULLY FINISHED")
print("=" * 60)

STEP 10 - BUILD COMPLETE ATTENDANCE REQUIRED MASTER

Loading source data...
Attendance source rows: 27711
Calendar rows: 365
Attendance source columns: 70

Required input columns: VALID

Standardizing keys and dates...
Key and date validation: VALID
Cost center classification: VALID
Weekly quota assignment: VALID

Building employee master...
Employee attributes detected: 57
Employee attribute columns: ['employee_name', 'day_s', 'shift', 'next_day_clock_time', 'total_work_hours', 'attendance_type', 'work_group', 'enter_overtime_awh', 'exit_overtime_awh', 'next_day_oawh', 'total_overtime_hours_awf', 'enter_overtime_bwh', 'exit_overtime_bwf', 'next_day_bwh', 'total_overtime_hours_bwh', 'total_overtime_hours', 'automatic_overtime', 'spl_no', 'status', 'attendance_source', 'attendance_edit_date', 'attendance_description', 'check_in_attendance_location_mob', 'check_out_attendance_location_mob', 'company_code', 'company_name', 'location_code', 'location_name', 'job_title_code', 'job_title_nam